# Память и время обучения

Бюджет памяти, активации и оценка по измеренному шагу.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

Память обучения включает веса, gradients, optimizer states, активации и временные buffers. FP32 AdamW часто требует примерно 16 байт на обучаемый параметр только для весов/градиентов/моментов; mixed precision и fused optimizers меняют эту сумму. Attention и длина контекста могут определять пик.

Ниже считаем нижнюю оценку для FP32 и измеряем CPU шаг. Это не прогноз для GPU и не обещание, что модель влезет. Для реального запуска используйте preflight и короткий benchmark на выбранном железе, с теми же batch, resolution, precision и optimizer. Начальные шаги включают прогрев, загрузку данных и compilation.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
import time, statistics
model = nn.Sequential(nn.Linear(64, 128), nn.GELU(), nn.Linear(128, 16))
optimizer = torch.optim.AdamW(model.parameters())
x = torch.randn(16, 64); target = torch.randn(16, 16)
count = sum(p.numel() for p in model.parameters())
print('FP32 weights + gradients + Adam moments, MiB:', count * 16 / 2**20)
durations = []
for step in range(15):
    begin = time.perf_counter()
    optimizer.zero_grad(set_to_none=True); loss = F.mse_loss(model(x), target)
    loss.backward(); optimizer.step()
    if step >= 5:
        durations.append(time.perf_counter() - begin)
median = statistics.median(durations)
print('measured CPU step, ms:', median * 1000)
print('1000 steps, compute only, seconds:', median * 1000)

## Попробуйте сами

Увеличьте batch в 4 раза и измерьте заново. Почему время не обязано вырасти ровно в 4 раза? Добавьте к бюджету памяти запас на активации и загрузку checkpoint.

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [PyTorch CUDA memory](https://docs.pytorch.org/docs/stable/notes/cuda.html#memory-management)